# MAGNET incremental TRACE viewer

Parses `MAGNET_INCREMENTAL_TRACE` logs and regenerates the HTML report.

Open the HTML in a browser for filters / timeline; use the plots below for a quick look in-notebook.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

REPO = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "runnable_code"))

from render_magnet_incremental_trace import parse_log, build_cycles, render_html

LOG = REPO / "outputs" / "magnet_incremental_trace_phase_c.log"
OUT = LOG.with_suffix(".html")

rows = parse_log(LOG.read_text(encoding="utf-8"))
cycles = build_cycles(rows)
OUT.write_text(render_html(rows, cycles, LOG.name), encoding="utf-8")
print(f"{len(rows)} TRACE lines → {len(cycles)} predict cycles")
print(f"HTML: {OUT}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
xs = [c["i"] for c in cycles]

axes[0, 0].plot(xs, [c["n_catalog"] for c in cycles], marker="o", label="n_catalog")
axes[0, 0].plot(xs, [c["n_candidates"] for c in cycles], marker="o", label="seis candidates")
axes[0, 0].plot(xs, [c["n_real"] for c in cycles], marker="o", label="recent n_real")
axes[0, 0].set_title("History / window sizes")
axes[0, 0].set_xlabel("predict cycle")
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(xs, [c["sampled_mag"] for c in cycles], marker="o", color="C1")
axes[0, 1].set_title("Sampled magnitude")
axes[0, 1].set_xlabel("predict cycle")

axes[1, 0].plot(xs, [c["seis_mean"] for c in cycles], marker="o", label="seis_mean")
axes[1, 0].plot(xs, [c["seis_max"] for c in cycles], marker="o", label="seis_max")
axes[1, 0].set_title("Seismicity feature stats")
axes[1, 0].set_xlabel("predict cycle")
axes[1, 0].legend(fontsize=8)

sc = axes[1, 1].scatter(
    [c["lng"] for c in cycles],
    [c["lat"] for c in cycles],
    c=xs,
    s=[40 + 25 * ((c["sampled_mag"] or 2) - 2) for c in cycles],
    cmap="viridis",
)
axes[1, 1].set_title("Query locations (color = cycle)")
axes[1, 1].set_xlabel("lng")
axes[1, 1].set_ylabel("lat")
fig.colorbar(sc, ax=axes[1, 1], label="cycle")
plt.show()

Regenerate after a new mock run:

```bash
python runnable_code/render_magnet_incremental_trace.py
# open outputs/magnet_incremental_trace_phase_c.html
```